# Stage 3 — Regressions & Cross-Tabs of `AttributionIndex_d`

Tests how the disaster-level Attribution Index (mean of article-level scores
$a_{i,d} \in \{-1, 0, 0.5, 1\}$, per `paper/attribution_codebook.md`) varies
with: disaster type, severity (cost/deaths), geography (coastal/inland,
red/blue state mix), time (year, pre/post-2010), and outlet characteristics
(national vs. local, political lean).

**Two units of analysis are used, deliberately:**

- **Disaster-level** (N = 209 disasters with ≥1 coded article) for type,
  severity, geography, and time — these are properties of the *event*, not
  of any individual article.
- **Article-level** (N = 1,252 relevant articles, standard errors clustered
  by disaster) for outlet characteristics — national/local status and
  political lean vary *within* a disaster (the same wildfire can be covered
  by both the AP and a local paper), so collapsing to one disaster-level
  number would throw away most of the signal. Clustering the SEs by disaster
  accounts for the fact that articles about the same disaster aren't
  independent observations.

**Important caveat:** the post-hoc audit (independent human-coded
reliability check, $\kappa$ and MAD vs. the LLM codes) has not yet been run.
Per the project owner, this analysis proceeds now and the audit will be
completed before the paper is submitted — these results should be treated
as provisional until that check clears.


## 0. Setup

In [1]:
import re
import warnings
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import openpyxl
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 130)
pd.set_option('display.max_columns', 25)

XLSX_PATH = Path('../attribution.xlsx')  # run notebook from code/


## 1. Reference classifications

Two fixed, documented classification rules are used for geography, since
`States` in `All Disasters` is a comma-separated, **alphabetically-ordered**
list for multi-state events (confirmed by inspection — it is NOT ordered by
impact, so a "primary state" heuristic is not usable). Instead, each
disaster gets a **share** measure: the fraction of its listed states that
are coastal / that voted Republican in 2020.

- **Coastal** = borders an ocean or the Gulf of Mexico (Atlantic, Pacific,
  Gulf, plus AK, HI, PR). Great-Lakes-only states (MI, OH, IN, IL, WI, MN,
  PA, NY-via-Erie) are classified as **inland** under this definition — a
  judgment call, flagged here rather than buried in code.
- **Red / Blue** = state-level winner of the 2020 presidential election
  (a single fixed snapshot, chosen because state partisan lean is reasonably
  stable and a moving target would be harder to defend than it's worth).
  8 of 209 covered disasters have `States = NATIONAL` (no state-level
  detail) or include Puerto Rico (a US territory with no 2020 presidential
  vote) — these get `NaN` red_share rather than a guessed value and are
  dropped from political-geography regressions only.


In [2]:
TRUMP_2020 = {'AL','AK','AR','FL','ID','IN','IA','KS','KY','LA','MS','MO','MT','NE',
              'NC','ND','OH','OK','SC','SD','TN','TX','UT','WV','WY'}
BIDEN_2020 = {'AZ','CA','CO','CT','DE','GA','HI','IL','MA','MD','ME','MI','MN','NV',
              'NH','NJ','NM','NY','OR','PA','RI','VA','VT','WA','WI'}

COASTAL = {'ME','NH','MA','RI','CT','NY','NJ','DE','MD','VA','NC','SC','GA','FL',
           'AL','MS','LA','TX','CA','OR','WA','AK','HI','PR'}

print(f"{len(TRUMP_2020)} states classified Trump-2020, {len(BIDEN_2020)} Biden-2020, "
      f"{len(COASTAL)} coastal states (incl. PR).")


25 states classified Trump-2020, 25 Biden-2020, 24 coastal states (incl. PR).


## 2. Build the analysis dataset

`AttributionIndex_d`, `N_d`, and `ContestedShare_d` are recomputed directly
from `Master` here rather than read from the `All Disasters` sheet's cached
formula values — those formulas (`=AVERAGEIFS(...)` etc.) were written by
`openpyxl`, which does not evaluate formulas, so Excel hasn't recalculated
them since the full LLM pass. Recomputing in pandas also makes the
construction of the index fully transparent in this notebook.


In [3]:
def get_domain(url):
    if not url:
        return None
    try:
        return urlparse(url).netloc.lower().replace('www.', '')
    except Exception:
        return None

def parse_adfontes_bias(text):
    if not text:
        return np.nan
    m = re.search(r'Bias\s*([+-]?\d+\.?\d*)', text)
    return float(m.group(1)) if m else np.nan

ALLSIDES_CATEGORY_FALLBACK = {'left': -5.0, 'lean left': -2.5, 'center': 0.0,
                               'lean right': 2.5, 'right': 5.0}

def parse_allsides_numeric(text):
    if not text:
        return np.nan
    m = re.search(r'\(([+-]?\d+\.?\d*)\s+on', text)
    return float(m.group(1)) if m else np.nan

def allsides_score(text):
    num = parse_allsides_numeric(text)
    if not np.isnan(num):
        return num
    if not text:
        return np.nan
    t = text.lower()
    for cat in ['lean left', 'lean right', 'left', 'right', 'center']:
        if t.startswith(cat):
            return ALLSIDES_CATEGORY_FALLBACK[cat]
    return np.nan

wb = openpyxl.load_workbook(XLSX_PATH, data_only=True)
ws, ad, dr = wb['Master'], wb['All Disasters'], wb['Domain Ratings']

# Domain -> political lean lookup
domain_lean = {}
for r in range(2, dr.max_row + 1):
    d = dr.cell(row=r, column=1).value
    if not d:
        continue
    als, af = dr.cell(row=r, column=4).value, dr.cell(row=r, column=5).value
    domain_lean[d] = {'adfontes_bias': parse_adfontes_bias(af),
                       'allsides_score': allsides_score(als)}

print(f"Domain Ratings: {len(domain_lean)} domains, "
      f"{sum(1 for v in domain_lean.values() if not np.isnan(v['adfontes_bias']))} with numeric Ad Fontes bias.")


Domain Ratings: 135 domains, 32 with numeric Ad Fontes bias.


In [4]:
# Article-level (Relevant rows only)
rows = []
for r in range(2, ws.max_row + 1):
    if ws.cell(row=r, column=15).value != 'Relevant':
        continue
    domain = get_domain(ws.cell(row=r, column=12).value)
    lean = domain_lean.get(domain, {})
    rows.append({
        'disaster_row': ws.cell(row=r, column=1).value,
        'attribution_score': ws.cell(row=r, column=16).value,
        'category': ws.cell(row=r, column=17).value,
        'contested': bool(ws.cell(row=r, column=19).value),
        'national': 1 if ws.cell(row=r, column=21).value == 'Yes' else 0,
        'domain': domain,
        'adfontes_bias': lean.get('adfontes_bias', np.nan),
        'allsides_score': lean.get('allsides_score', np.nan),
    })
articles = pd.DataFrame(rows)
print(f"Relevant, coded articles: {len(articles)}")
articles.head()


Relevant, coded articles: 1252


,disaster_row,attribution_score,category,contested,national,domain,adfontes_bias,allsides_score
0,1,0.0,C,False,1,nytimes.com,-8.02,-2.2
1,3,0.0,C,False,1,cnn.com,-6.27,-1.3
2,3,0.0,C,False,1,cnn.com,-6.27,-1.3
3,3,-1.0,D,False,1,cnn.com,-6.27,-1.3
4,5,-1.0,D,False,1,baltimoresun.com,-3.79,0.0


In [5]:
# Disaster-level base table
drows = []
for r in range(2, ad.max_row + 1):
    drows.append({
        'disaster_row': ad.cell(row=r, column=1).value,
        'name': ad.cell(row=r, column=2).value,
        'disaster_type': ad.cell(row=r, column=3).value,
        'begin_date': ad.cell(row=r, column=4).value,
        'end_date': ad.cell(row=r, column=5).value,
        'cost_cpi': ad.cell(row=r, column=6).value,
        'deaths': ad.cell(row=r, column=8).value,
        'states_raw': ad.cell(row=r, column=9).value,
    })
disasters = pd.DataFrame(drows)

def parse_date(d):
    if d is None:
        return None
    s = str(int(d))
    return pd.Timestamp(year=int(s[:4]), month=int(s[4:6]), day=int(s[6:8]))

disasters['begin_dt'] = disasters['begin_date'].apply(parse_date)
disasters['end_dt'] = disasters['end_date'].apply(parse_date)
disasters['year'] = disasters['begin_dt'].dt.year
disasters['year_c'] = disasters['year'] - disasters['year'].min()  # centered, for cleaner regression output
disasters['era'] = np.where(disasters['year'] < 2010, 'pre-2010', 'post-2010')
disasters['duration_days'] = (disasters['end_dt'] - disasters['begin_dt']).dt.days.clip(lower=0) + 1

def state_list(s):
    return [x.strip() for x in s.split(',')] if s else []

def coastal_share(states):
    return np.mean([s in COASTAL for s in states]) if states else np.nan

def red_share(states):
    classified = [s for s in states if s in TRUMP_2020 or s in BIDEN_2020]
    return np.mean([s in TRUMP_2020 for s in classified]) if classified else np.nan

disasters['states_list'] = disasters['states_raw'].apply(state_list)
disasters['n_states'] = disasters['states_list'].apply(len)
disasters['coastal_share'] = disasters['states_list'].apply(coastal_share)
disasters['red_share'] = disasters['states_list'].apply(red_share)
disasters['any_coastal'] = np.where(disasters['coastal_share'] > 0, 'Coastal', 'Inland')
disasters['political_cat'] = np.where(disasters['red_share'] > 0.5, 'Majority Red',
                                np.where(disasters['red_share'] < 0.5, 'Majority Blue', 'Split/Tied'))

agg = articles.groupby('disaster_row').agg(
    N_d=('attribution_score', 'size'),
    AttributionIndex_d=('attribution_score', 'mean'),
    ContestedShare_d=('contested', 'mean'),
    NationalShare_d=('national', 'mean'),
).reset_index()

disasters = disasters.merge(agg, on='disaster_row', how='left')
disasters['N_d'] = disasters['N_d'].fillna(0).astype(int)
disasters['log_cost'] = np.log(disasters['cost_cpi'])
disasters['log_deaths'] = np.log(disasters['deaths'] + 1)
disasters['log_duration'] = np.log(disasters['duration_days'])

covered = disasters[disasters['N_d'] >= 1].copy()
uncovered = disasters[disasters['N_d'] == 0].copy()

print(f"Total NOAA disasters (2000-2024): {len(disasters)}")
print(f"Covered (>=1 relevant coded article): {len(covered)}")
print(f"Uncovered (N_d = 0, analyzed separately, not dropped): {len(uncovered)}")
print(f"\nGeography unclassifiable (States='NATIONAL' or territory only): "
      f"{covered['red_share'].isna().sum()} of {len(covered)} covered disasters")


Total NOAA disasters (2000-2024): 313
Covered (>=1 relevant coded article): 209
Uncovered (N_d = 0, analyzed separately, not dropped): 104

Geography unclassifiable (States='NATIONAL' or territory only): 8 of 209 covered disasters


## A.0 Figure: Distribution of the Disaster-Level Attribution Index

Before breaking the index down by regressor, a look at its overall shape
across the 209 covered disasters: where the mass sits, and how much of it
is exactly zero (no causal claim made, category C, the modal outcome).
Saved to `../paper/figures/attribution_index_hist.pdf` for use in the
paper.


In [6]:
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.hist(covered['AttributionIndex_d'], bins=np.arange(-1.05, 1.15, 0.1),
        color='#2b5b84', edgecolor='white', alpha=0.85)
ax.axvline(0, color='gray', linestyle=':', linewidth=1)
ax.axvline(covered['AttributionIndex_d'].mean(), color='#b5402a', linestyle='-', linewidth=1.5,
           label=f"Mean = {covered['AttributionIndex_d'].mean():.3f}")
ax.set_xlabel('AttributionIndex$_d$')
ax.set_ylabel('Number of disasters')
ax.set_title(f'Distribution of Disaster-Level Attribution Index (N={len(covered)})')
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
fig.savefig('../paper/figures/attribution_index_hist.pdf')
plt.show()

print(f"N = {len(covered)}")
print(f"Mean = {covered['AttributionIndex_d'].mean():.4f}, Median = {covered['AttributionIndex_d'].median():.4f}, "
      f"SD = {covered['AttributionIndex_d'].std():.4f}")
print(f"Share exactly 0.0: {(covered['AttributionIndex_d'] == 0).mean():.1%}")
print(f"Share > 0: {(covered['AttributionIndex_d'] > 0).mean():.1%}, "
      f"Share < 0: {(covered['AttributionIndex_d'] < 0).mean():.1%}")


N = 209
Mean = 0.0281, Median = 0.0000, SD = 0.2410
Share exactly 0.0: 55.5%
Share > 0: 30.6%, Share < 0: 13.9%


## A. Disaster Type

Cross-tab of mean/median/SD by NOAA disaster type, a one-way ANOVA, and an
OLS with disaster-type dummies (reference category = Severe Storm, the
modal type).


In [7]:
tab_type = covered.groupby('disaster_type')['AttributionIndex_d'].agg(
    N='count', Mean='mean', Median='median', SD='std').sort_values('N', ascending=False)
print("Mean AttributionIndex_d by disaster type:")
print(tab_type.round(3))

groups = [g['AttributionIndex_d'].values for _, g in covered.groupby('disaster_type')]
f_stat, p_val = stats.f_oneway(*groups)
print(f"\nOne-way ANOVA across {len(groups)} disaster types: F = {f_stat:.3f}, p = {p_val:.4f}")

m_type = smf.ols(
    "AttributionIndex_d ~ C(disaster_type, Treatment(reference='Severe Storm'))",
    data=covered).fit(cov_type='HC1')
print("\nOLS, AttributionIndex_d ~ disaster type dummies (ref = Severe Storm), robust SE:")
print(m_type.summary())


Mean AttributionIndex_d by disaster type:
                   N   Mean  Median     SD
disaster_type                             
Severe Storm      89  0.005   0.000  0.236
Tropical Cyclone  45  0.055   0.022  0.113
Flooding          25  0.028   0.000  0.175
Drought           18  0.026   0.017  0.476
Wildfire          18  0.048   0.021  0.315
Winter Storm      12  0.074   0.000  0.148
Freeze             2  0.000   0.000  0.000

One-way ANOVA across 7 disaster types: F = 0.321, p = 0.9254

OLS, AttributionIndex_d ~ disaster type dummies (ref = Severe Storm), robust SE:
                            OLS Regression Results                            
Dep. Variable:     AttributionIndex_d   R-squared:                       0.009
Model:                            OLS   Adj. R-squared:                 -0.020
Method:                 Least Squares   F-statistic:                     2.508
Date:                Mon, 22 Jun 2026   Prob (F-statistic):             0.0231
Time:                        03:

## B. Severity (cost, deaths, duration)

CPI-adjusted cost and deaths are both heavily right-skewed (a handful of
multi-billion-dollar, mass-casualty events), so both are log-transformed
(`log(deaths + 1)` to handle zero-death disasters). Event duration is added
as a bonus regressor — sustained, multi-week disasters (e.g. droughts) may
draw different framing than single-day tornado outbreaks regardless of
cost/deaths.


In [8]:
print("Correlation matrix:")
print(covered[['AttributionIndex_d', 'log_cost', 'log_deaths', 'log_duration']].corr().round(3))

m_sev = smf.ols("AttributionIndex_d ~ log_cost + log_deaths + log_duration",
                data=covered).fit(cov_type='HC1')
print("\nOLS, AttributionIndex_d ~ log(cost) + log(deaths+1) + log(duration), robust SE:")
print(m_sev.summary())


Correlation matrix:
                    AttributionIndex_d  log_cost  log_deaths  log_duration
AttributionIndex_d               1.000     0.091       0.074         0.002
log_cost                         0.091     1.000       0.680         0.237
log_deaths                       0.074     0.680       1.000         0.178
log_duration                     0.002     0.237       0.178         1.000

OLS, AttributionIndex_d ~ log(cost) + log(deaths+1) + log(duration), robust SE:
                            OLS Regression Results                            
Dep. Variable:     AttributionIndex_d   R-squared:                       0.009
Model:                            OLS   Adj. R-squared:                 -0.006
Method:                 Least Squares   F-statistic:                     1.327
Date:                Mon, 22 Jun 2026   Prob (F-statistic):              0.267
Time:                        03:26:36   Log-Likelihood:                 2.2974
No. Observations:                 209   AIC:      

## C. Geography (coastal/inland, red/blue states)

Cross-tabs use the simple binary version (`any_coastal`, `political_cat`,
majority-rule); the regression uses the continuous share versions, which
preserve information for multi-state disasters instead of collapsing them
to a single category. `NaN`-geography disasters (States = NATIONAL /
territory-only) are dropped from this section only.


In [9]:
geo = covered.dropna(subset=['coastal_share', 'red_share']).copy()
print(f"N with classifiable geography: {len(geo)} of {len(covered)}\n")

print("-- Coastal vs Inland (any coastal state present) --")
print(geo.groupby('any_coastal')['AttributionIndex_d'].agg(N='count', Mean='mean', SD='std').round(3))
t_geo, p_geo = stats.ttest_ind(geo.loc[geo.any_coastal == 'Coastal', 'AttributionIndex_d'],
                                geo.loc[geo.any_coastal == 'Inland', 'AttributionIndex_d'],
                                equal_var=False)
print(f"Welch t-test Coastal vs Inland: t = {t_geo:.3f}, p = {p_geo:.4f}")

print("\n-- Majority Red vs Majority Blue vs Split (2020 pres. vote, by state count) --")
print(geo.groupby('political_cat')['AttributionIndex_d'].agg(N='count', Mean='mean', SD='std').round(3))

m_geo = smf.ols("AttributionIndex_d ~ coastal_share + red_share", data=geo).fit(cov_type='HC1')
print("\nOLS, AttributionIndex_d ~ coastal_share + red_share (continuous, 0-1), robust SE:")
print(m_geo.summary())


N with classifiable geography: 201 of 209

-- Coastal vs Inland (any coastal state present) --
               N   Mean     SD
any_coastal                   
Coastal      179  0.023  0.235
Inland        22  0.070  0.301
Welch t-test Coastal vs Inland: t = -0.715, p = 0.4812

-- Majority Red vs Majority Blue vs Split (2020 pres. vote, by state count) --
                 N   Mean     SD
political_cat                   
Majority Blue   71  0.023  0.247
Majority Red   125  0.023  0.239
Split/Tied       5  0.213  0.250

OLS, AttributionIndex_d ~ coastal_share + red_share (continuous, 0-1), robust SE:
                            OLS Regression Results                            
Dep. Variable:     AttributionIndex_d   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                 -0.009
Method:                 Least Squares   F-statistic:                    0.1630
Date:                Mon, 22 Jun 2026   Prob (F-statistic):              0.850
Tim

## D. Time (year trend, pre/post-2010)

Pre/post-2010 mirrors the era split already used for stratifying the
calibration and audit samples elsewhere in the project.


In [10]:
print(covered.groupby('era')['AttributionIndex_d'].agg(N='count', Mean='mean', SD='std').round(3))
t_era, p_era = stats.ttest_ind(covered.loc[covered.era == 'pre-2010', 'AttributionIndex_d'],
                                covered.loc[covered.era == 'post-2010', 'AttributionIndex_d'],
                                equal_var=False)
print(f"\nWelch t-test pre- vs post-2010: t = {t_era:.3f}, p = {p_era:.4f}")

m_time = smf.ols("AttributionIndex_d ~ year_c", data=covered).fit(cov_type='HC1')
print("\nOLS, AttributionIndex_d ~ year (centered at 2000), robust SE:")
print(m_time.summary())
print(f"\nImplied annual increase in AttributionIndex_d: {m_time.params['year_c']:.4f} per year "
      f"(p = {m_time.pvalues['year_c']:.4f})")


             N   Mean    SD
era                        
post-2010  172  0.057  0.23
pre-2010    37 -0.105  0.25

Welch t-test pre- vs post-2010: t = -3.628, p = 0.0007

OLS, AttributionIndex_d ~ year (centered at 2000), robust SE:
                            OLS Regression Results                            
Dep. Variable:     AttributionIndex_d   R-squared:                       0.125
Model:                            OLS   Adj. R-squared:                  0.120
Method:                 Least Squares   F-statistic:                     20.69
Date:                Mon, 22 Jun 2026   Prob (F-statistic):           9.17e-06
Time:                        03:26:36   Log-Likelihood:                 15.263
No. Observations:                 209   AIC:                            -26.53
Df Residuals:                     207   BIC:                            -19.84
Df Model:                           1                                         
Covariance Type:                  HC1                     

## E. Outlet Type (National vs. Local)

Two versions: a disaster-level test using `NationalShare_d` (share of each
disaster's coverage from national outlets, via the existing `National
Newspaper` column in `Master`), and the more statistically appropriate
article-level test with standard errors clustered by disaster (since
articles about the same disaster aren't independent draws, and national vs.
local status varies *within* most disasters).


In [11]:
print("-- Disaster-level: AttributionIndex_d ~ NationalShare_d --")
m_nat_d = smf.ols("AttributionIndex_d ~ NationalShare_d", data=covered).fit(cov_type='HC1')
print(m_nat_d.summary().tables[1])

print("\n-- Article-level: attribution_score ~ national, clustered SE by disaster --")
print(articles.groupby('national')['attribution_score'].agg(N='count', Mean='mean', SD='std').round(3))
m_nat_a = smf.ols("attribution_score ~ national", data=articles).fit(
    cov_type='cluster', cov_kwds={'groups': articles['disaster_row']})
print(m_nat_a.summary())


-- Disaster-level: AttributionIndex_d ~ NationalShare_d --
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept           0.0246      0.041      0.604      0.546      -0.055       0.104
NationalShare_d     0.0055      0.057      0.096      0.923      -0.107       0.118

-- Article-level: attribution_score ~ national, clustered SE by disaster --
            N   Mean     SD
national                   
0         549  0.077  0.337
1         703  0.120  0.414
                            OLS Regression Results                            
Dep. Variable:      attribution_score   R-squared:                       0.003
Model:                            OLS   Adj. R-squared:                  0.002
Method:                 Least Squares   F-statistic:                     3.085
Date:                Mon, 22 Jun 2026   Prob (F-statistic):             0.0805
Time:                

## F. Outlet Political Lean

Viable, but **article-level only** — only 33 of the 135 distinct domains in
`Master` have an individual AllSides/Ad Fontes rating page (the rest are
small local papers that neither service rates individually). Those 33
domains cover a high share of *articles* though, because coverage is
concentrated in large outlets: 71% of relevant articles come from a
rated domain.

Ad Fontes' numeric "Bias" score (-42 to +42, more positive = more right-
leaning) is used as the primary continuous measure because every rated
domain has a clean numeric value; AllSides is reported as a robustness
check (some AllSides entries only had a categorical rating with no exact
meter value, so those were mapped to a fixed numeric scale: Left=-5, Lean
Left=-2.5, Center=0, Lean Right=+2.5, Right=+5 — a coarser approximation
than Ad Fontes' native numbers).


In [12]:
sub_af = articles.dropna(subset=['adfontes_bias'])
print(f"-- Ad Fontes bias score, N = {len(sub_af)} of {len(articles)} articles ({len(sub_af)/len(articles):.1%}) --")
m_lean_af = smf.ols("attribution_score ~ adfontes_bias", data=sub_af).fit(
    cov_type='cluster', cov_kwds={'groups': sub_af['disaster_row']})
print(m_lean_af.summary())

sub_as = articles.dropna(subset=['allsides_score'])
print(f"\n-- Robustness: AllSides score, N = {len(sub_as)} of {len(articles)} articles --")
m_lean_as = smf.ols("attribution_score ~ allsides_score", data=sub_as).fit(
    cov_type='cluster', cov_kwds={'groups': sub_as['disaster_row']})
print(m_lean_as.summary().tables[1])


-- Ad Fontes bias score, N = 889 of 1252 articles (71.0%) --
                            OLS Regression Results                            
Dep. Variable:      attribution_score   R-squared:                       0.007
Model:                            OLS   Adj. R-squared:                  0.006
Method:                 Least Squares   F-statistic:                     10.02
Date:                Mon, 22 Jun 2026   Prob (F-statistic):            0.00180
Time:                        03:26:36   Log-Likelihood:                -457.91
No. Observations:                 889   AIC:                             919.8
Df Residuals:                     887   BIC:                             929.4
Df Model:                           1                                         
Covariance Type:              cluster                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------

## F.1 Figure: Outlet Lean vs. Article Attribution Score

The article-level attribution score only takes four values
($\{-1, 0, 0.5, 1\}$), so a raw scatter would be almost entirely
overplotted. This jitters the y-axis slightly for visibility, overlays the
OLS fit (same model as Section F), and adds binned means (8 equal-width
bins of Ad Fontes Bias score) as a model-free check on the linear fit.
Saved to `../paper/figures/outlet_lean_scatter.pdf`.


In [13]:
sub_scatter = articles.dropna(subset=['adfontes_bias']).copy()
rng = np.random.default_rng(42)
jitter = rng.uniform(-0.05, 0.05, size=len(sub_scatter))

fig, ax = plt.subplots(figsize=(6.5, 4.3))
ax.scatter(sub_scatter['adfontes_bias'], sub_scatter['attribution_score'] + jitter,
           alpha=0.18, s=14, color='#2b5b84', linewidths=0, label='Articles (jittered)')

m_scatter = smf.ols("attribution_score ~ adfontes_bias", data=sub_scatter).fit(
    cov_type='cluster', cov_kwds={'groups': sub_scatter['disaster_row']})
xs = np.linspace(sub_scatter['adfontes_bias'].min(), sub_scatter['adfontes_bias'].max(), 100)
ys = m_scatter.params['Intercept'] + m_scatter.params['adfontes_bias'] * xs
ax.plot(xs, ys, color='#b5402a', linewidth=2, label='OLS fit (clustered SE)')

bins = pd.cut(sub_scatter['adfontes_bias'], bins=8)
binned_means = sub_scatter.groupby(bins, observed=True)['attribution_score'].mean()
bin_centers = sub_scatter.groupby(bins, observed=True)['adfontes_bias'].mean()
ax.scatter(bin_centers, binned_means, color='#1a1a1a', s=45, zorder=5, marker='D',
           label='Binned mean (8 bins)')

ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)
ax.set_xlabel('Outlet Ad Fontes Bias score (more positive = more right-leaning)')
ax.set_ylabel('Article attribution score $a_{i,d}$ (jittered)')
ax.set_title('Outlet Political Lean vs. Article Attribution Score')
ax.legend(frameon=False, fontsize=9, loc='upper right')
fig.tight_layout()
fig.savefig('../paper/figures/outlet_lean_scatter.pdf')
plt.show()

print(f"N = {len(sub_scatter)}")
print(f"OLS: b = {m_scatter.params['adfontes_bias']:.4f}, p = {m_scatter.pvalues['adfontes_bias']:.4f}")
print("Binned means by Ad Fontes Bias score bin:")
print(pd.DataFrame({'bin_center': bin_centers.round(1), 'mean_score': binned_means.round(3)}).to_string(index=False))


N = 889
OLS: b = -0.0132, p = 0.0016
Binned means by Ad Fontes Bias score bin:
 bin_center  mean_score
       -7.2       0.159
       -4.2       0.053
       -2.6       0.095
       -0.8       0.017
        2.1       0.200
        3.7       0.500
        5.4       0.500
        9.5       0.000


## G. Coverage Volume (bonus regressor: $N_d$)

Not in the original request, but worth checking on its own: does the
*amount* of coverage a disaster receives correlate with how it's framed?
(E.g., high-profile disasters might draw more climate-attribution
commentary, or conversely more routine wire coverage that dilutes toward
neutral framing.)


In [14]:
m_nd = smf.ols("AttributionIndex_d ~ N_d", data=covered).fit(cov_type='HC1')
print(m_nd.summary().tables[1])
print(f"\nN_d distribution among covered disasters: "
      f"median={covered['N_d'].median():.0f}, mean={covered['N_d'].mean():.1f}, max={covered['N_d'].max()}")


                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.0551      0.027     -2.054      0.040      -0.108      -0.003
N_d            0.0139      0.003      4.664      0.000       0.008       0.020

N_d distribution among covered disasters: median=4, mean=6.0, max=27


## H. Combined Multivariate Model — Disaster Level

All disaster-level regressors together. Coverage volume ($N_d$) is included
as a control. Geography-unclassifiable disasters (States=NATIONAL/
territory-only) are dropped here (listwise), same as in Section C.


In [15]:
geo_full = covered.dropna(subset=['coastal_share', 'red_share'])
m_combined_d = smf.ols(
    "AttributionIndex_d ~ C(disaster_type, Treatment(reference='Severe Storm')) + "
    "log_cost + log_deaths + log_duration + coastal_share + red_share + year_c + N_d",
    data=geo_full).fit(cov_type='HC1')
print(f"N = {int(m_combined_d.nobs)}")
print(m_combined_d.summary())


N = 201


                            OLS Regression Results                            
Dep. Variable:     AttributionIndex_d   R-squared:                       0.201
Model:                            OLS   Adj. R-squared:                  0.145
Method:                 Least Squares   F-statistic:                     3.941
Date:                Mon, 22 Jun 2026   Prob (F-statistic):           1.14e-05
Time:                        03:26:37   Log-Likelihood:                 22.665
No. Observations:                 201   AIC:                            -17.33
Df Residuals:                     187   BIC:                             28.92
Df Model:                          13                                         
Covariance Type:                  HC1                                         
                                                                                coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------

## I. Combined Multivariate Model — Article Level (clustered SE)

All article-level-viable regressors together — outlet type, outlet
political lean, disaster type, and year — with standard errors clustered by
disaster. Restricted to the 71% of articles with a rated domain (since
political lean isn't available otherwise); this is the model to read if the
question is "controlling for everything else, does outlet lean or
national/local status independently predict framing?"


In [16]:
art_full = articles.merge(
    covered[['disaster_row', 'disaster_type', 'year_c', 'coastal_share', 'red_share']],
    on='disaster_row', how='left'
).dropna(subset=['adfontes_bias'])

m_combined_a = smf.ols(
    "attribution_score ~ national + adfontes_bias + "
    "C(disaster_type, Treatment(reference='Severe Storm')) + year_c",
    data=art_full).fit(cov_type='cluster', cov_kwds={'groups': art_full['disaster_row']})
print(f"N = {int(m_combined_a.nobs)} articles, clustered on {art_full['disaster_row'].nunique()} disasters")
print(m_combined_a.summary())


N = 889 articles, clustered on 194 disasters
                            OLS Regression Results                            
Dep. Variable:      attribution_score   R-squared:                       0.124
Model:                            OLS   Adj. R-squared:                  0.115
Method:                 Least Squares   F-statistic:                     8.062
Date:                Mon, 22 Jun 2026   Prob (F-statistic):           4.07e-10
Time:                        03:26:37   Log-Likelihood:                -402.18
No. Observations:                 889   AIC:                             824.4
Df Residuals:                     879   BIC:                             872.3
Df Model:                           9                                         
Covariance Type:              cluster                                         
                                                                                coef    std err          z      P>|z|      [0.025      0.975]
-----------------------

## J. Summary

Bivariate results at a glance (disaster-level unless noted). This is a
quick-reference table only — see each section above for full model output,
N, and robust/clustered SEs.

**Reminder:** these results are provisional pending the post-hoc audit
(independent human-coded $\kappa$/MAD check against the LLM codes) — see
`paper/attribution_codebook.md`, Step 5. Treat statistically significant
findings here as suggestive until that check clears, especially anything
resting on the B/C or C/E category boundaries, which the codebook itself
flags as the fuzziest.


In [17]:
summary_rows = [
    ("Disaster type (ANOVA)", f"F={f_stat:.2f}", p_val),
    ("log(cost)", f"b={m_sev.params['log_cost']:.4f}", m_sev.pvalues['log_cost']),
    ("log(deaths+1)", f"b={m_sev.params['log_deaths']:.4f}", m_sev.pvalues['log_deaths']),
    ("log(duration)", f"b={m_sev.params['log_duration']:.4f}", m_sev.pvalues['log_duration']),
    ("Coastal vs Inland (t-test)", f"t={t_geo:.2f}", p_geo),
    ("red_share (continuous)", f"b={m_geo.params['red_share']:.4f}", m_geo.pvalues['red_share']),
    ("Year trend", f"b={m_time.params['year_c']:.4f}/yr", m_time.pvalues['year_c']),
    ("National share (disaster-level)", f"b={m_nat_d.params['NationalShare_d']:.4f}", m_nat_d.pvalues['NationalShare_d']),
    ("National (article-level, clustered)", f"b={m_nat_a.params['national']:.4f}", m_nat_a.pvalues['national']),
    ("Ad Fontes bias (article-level, clustered)", f"b={m_lean_af.params['adfontes_bias']:.4f}", m_lean_af.pvalues['adfontes_bias']),
    ("Coverage volume N_d", f"b={m_nd.params['N_d']:.4f}", m_nd.pvalues['N_d']),
]
summary_df = pd.DataFrame(summary_rows, columns=['Regressor', 'Estimate', 'p-value'])
summary_df['Sig.'] = summary_df['p-value'].apply(
    lambda p: '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else '')
print(summary_df.to_string(index=False))


                                Regressor    Estimate  p-value Sig.
                    Disaster type (ANOVA)      F=0.32 0.925430     
                                log(cost)    b=0.0172 0.326608     
                            log(deaths+1)    b=0.0035 0.821645     
                            log(duration)   b=-0.0030 0.838473     
               Coastal vs Inland (t-test)     t=-0.72 0.481154     
                   red_share (continuous)    b=0.0167 0.737524     
                               Year trend b=0.0129/yr 0.000005  ***
          National share (disaster-level)    b=0.0055 0.923421     
      National (article-level, clustered)    b=0.0437 0.078998    *
Ad Fontes bias (article-level, clustered)   b=-0.0132 0.001550  ***
                      Coverage volume N_d    b=0.0139 0.000003  ***


## K. Robustness Checks

Stress-tests of the two headline findings from Sections D and F -- the year
trend and the outlet-political-lean effect -- against alternative
specifications, sample restrictions, and a placebo. Each check is its own
cell so results can be read individually; a summary table is at the end
(Section K.8).


### K.1 Disaster Fixed Effects (strongest test of the outlet-lean result)

The combined article-level model in Section I controls for disaster type
and year, but standard errors are only *clustered* by disaster -- any
other disaster-level confound could still leak into the national/lean
comparison. This check absorbs disaster fixed effects entirely (via
within-disaster demeaning, equivalent to including a dummy for every
disaster), so `national` and `adfontes_bias` are identified purely from
variation **within the same disaster** -- e.g., the AP vs. a local paper
covering the same wildfire. Disasters with only one lean-rated article
contribute no within-disaster variation and drop out automatically.


In [18]:
sub_fe = articles.dropna(subset=['adfontes_bias']).copy()
sub_fe['grp_n'] = sub_fe.groupby('disaster_row')['disaster_row'].transform('size')
multi = sub_fe[sub_fe['grp_n'] >= 2].copy()
print(f"Articles with lean data: {len(sub_fe)}; in disasters with >=2 such articles: "
      f"{len(multi)} ({multi['disaster_row'].nunique()} disasters contribute identifying variation)")

for col in ['attribution_score', 'national', 'adfontes_bias']:
    multi[col + '_dm'] = multi[col] - multi.groupby('disaster_row')[col].transform('mean')

m_fe = smf.ols("attribution_score_dm ~ national_dm + adfontes_bias_dm - 1", data=multi).fit(
    cov_type='cluster', cov_kwds={'groups': multi['disaster_row']})
print("\nWithin-disaster (fixed-effects) model -- variables are demeaned by disaster, no intercept:")
print(m_fe.summary())


Articles with lean data: 889; in disasters with >=2 such articles: 839 (144 disasters contribute identifying variation)

Within-disaster (fixed-effects) model -- variables are demeaned by disaster, no intercept:
                                  OLS Regression Results                                 
Dep. Variable:     attribution_score_dm   R-squared (uncentered):                   0.017
Model:                              OLS   Adj. R-squared (uncentered):              0.014
Method:                   Least Squares   F-statistic:                              4.204
Date:                  Mon, 22 Jun 2026   Prob (F-statistic):                      0.0168
Time:                          03:26:37   Log-Likelihood:                         -245.96
No. Observations:                   839   AIC:                                      495.9
Df Residuals:                       837   BIC:                                      505.4
Df Model:                             2                             

### K.2 Binary Attribution Measure

The headline results treat $a_{i,d} \in \{-1, 0, 0.5, 1\}$ as cardinal.
This collapses to a coarser, ordinal-free binary: "any climate mention"
(categories A or B) vs. "none" (C, D, or E). If the year trend and
outlet-lean results are artifacts of the specific $\{-1,0,0.5,1\}$ scaling
rather than the underlying pattern, they should weaken or disappear here.


In [19]:
articles['any_climate'] = articles['category'].isin(['A', 'B']).astype(int)

any_climate_share = articles.groupby('disaster_row')['any_climate'].mean().rename('AnyClimateShare_d')
cov_bin = covered.merge(any_climate_share, left_on='disaster_row', right_index=True, how='left')
m_bin_time = smf.ols("AnyClimateShare_d ~ year_c", data=cov_bin).fit(cov_type='HC1')
print("-- Disaster-level year trend, binary DV (share of articles with any climate mention) --")
print(m_bin_time.summary().tables[1])

sub_bin = articles.dropna(subset=['adfontes_bias'])
m_bin_lean = smf.ols("any_climate ~ adfontes_bias", data=sub_bin).fit(
    cov_type='cluster', cov_kwds={'groups': sub_bin['disaster_row']})
print("\n-- Article-level outlet lean, binary DV, clustered SE --")
print(m_bin_lean.summary().tables[1])


-- Disaster-level year trend, binary DV (share of articles with any climate mention) --
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.0387      0.025     -1.554      0.120      -0.088       0.010
year_c         0.0100      0.002      5.320      0.000       0.006       0.014

-- Article-level outlet lean, binary DV, clustered SE --
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept         0.1349      0.037      3.634      0.000       0.062       0.208
adfontes_bias    -0.0177      0.005     -3.787      0.000      -0.027      -0.009


### K.3 D-Subtype Recoding

Category D collapses two conceptually different framings to the same
$-1.0$ score: attributing the event to *natural variability* ("just
weather," 33 articles) vs. to *other human causes* (land use,
infrastructure, development -- 20 articles). The latter isn't really a
climate-skeptical frame, just a non-climate one, so this check rescales
`other_human_causes` D-articles to $0.0$ (same as "no claim made") and
leaves `natural_variability` D-articles at $-1.0$, then re-tests both
headline results.


In [20]:
d_subtype = []
for r in range(2, ws.max_row + 1):
    if ws.cell(row=r, column=15).value != 'Relevant':
        continue
    d_subtype.append(ws.cell(row=r, column=18).value)
articles['d_subtype'] = d_subtype
articles['score_alt'] = np.where(articles['d_subtype'] == 'other_human_causes',
                                  0.0, articles['attribution_score'])
print(f"Rows recoded (other_human_causes D, -1 -> 0): {(articles['score_alt'] != articles['attribution_score']).sum()}")

alt_index = articles.groupby('disaster_row')['score_alt'].mean().rename('AttributionIndex_d_alt')
cov_alt = covered.merge(alt_index, left_on='disaster_row', right_index=True, how='left')
m_alt_time = smf.ols("AttributionIndex_d_alt ~ year_c", data=cov_alt).fit(cov_type='HC1')
print("\n-- Disaster-level year trend, D-subtype-recoded DV --")
print(m_alt_time.summary().tables[1])

sub_alt = articles.dropna(subset=['adfontes_bias'])
m_alt_lean = smf.ols("score_alt ~ adfontes_bias", data=sub_alt).fit(
    cov_type='cluster', cov_kwds={'groups': sub_alt['disaster_row']})
print("\n-- Article-level outlet lean, D-subtype-recoded DV, clustered SE --")
print(m_alt_lean.summary().tables[1])


Rows recoded (other_human_causes D, -1 -> 0): 20

-- Disaster-level year trend, D-subtype-recoded DV --
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.1231      0.042     -2.916      0.004      -0.206      -0.040
year_c         0.0102      0.002      4.133      0.000       0.005       0.015

-- Article-level outlet lean, D-subtype-recoded DV, clustered SE --
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept         0.0607      0.033      1.845      0.065      -0.004       0.125
adfontes_bias    -0.0135      0.004     -3.389      0.001      -0.021      -0.006


### K.4 Excluding Contested (E) Articles

Category E (mixed/contested) scores $0$, identically to "no claim made"
(C) -- a modeling choice the codebook flags explicitly as a simplification.
This check drops all 10 category-E articles entirely (rather than scoring
them) and re-tests both headline results on the resulting sample.


In [21]:
articles_noE = articles[articles['category'] != 'E'].copy()
print(f"Articles excluded: {(articles['category'] == 'E').sum()}; remaining: {len(articles_noE)}")

agg_noE = articles_noE.groupby('disaster_row').agg(
    N_d=('attribution_score', 'size'), AttributionIndex_d=('attribution_score', 'mean'))
cov_noE = covered[['disaster_row', 'disaster_type', 'year_c']].merge(agg_noE, on='disaster_row', how='inner')
print(f"Disasters retaining >=1 article after exclusion: {len(cov_noE)} (of {len(covered)})")

m_noE_time = smf.ols("AttributionIndex_d ~ year_c", data=cov_noE).fit(cov_type='HC1')
print("\n-- Disaster-level year trend, E excluded --")
print(m_noE_time.summary().tables[1])

sub_noE = articles_noE.dropna(subset=['adfontes_bias'])
m_noE_lean = smf.ols("attribution_score ~ adfontes_bias", data=sub_noE).fit(
    cov_type='cluster', cov_kwds={'groups': sub_noE['disaster_row']})
print("\n-- Article-level outlet lean, E excluded, clustered SE --")
print(m_noE_lean.summary().tables[1])


Articles excluded: 10; remaining: 1242
Disasters retaining >=1 article after exclusion: 209 (of 209)

-- Disaster-level year trend, E excluded --
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.1851      0.050     -3.683      0.000      -0.284      -0.087
year_c         0.0130      0.003      4.590      0.000       0.007       0.019

-- Article-level outlet lean, E excluded, clustered SE --
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept         0.0487      0.033      1.469      0.142      -0.016       0.114
adfontes_bias    -0.0135      0.004     -3.195      0.001      -0.022      -0.005


### K.5 Restricting to $N_d \geq 3$

The disaster-level index can be a single article's score for low-coverage
disasters, which is noisy. This restricts the year-trend test to the 138
disasters with at least 3 coded articles, where the index is a more
stable average.


In [22]:
cov_n3 = covered[covered['N_d'] >= 3]
print(f"Disasters with N_d >= 3: {len(cov_n3)} (of {len(covered)})")

m_n3 = smf.ols("AttributionIndex_d ~ year_c", data=cov_n3).fit(cov_type='HC1')
print(m_n3.summary().tables[1])


Disasters with N_d >= 3: 138 (of 209)
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.1317      0.034     -3.894      0.000      -0.198      -0.065
year_c         0.0112      0.002      5.651      0.000       0.007       0.015


### K.6 Restricting the Outlet-Lean Test to Post-2010 Only

The pre-2010 era has only 37 covered disasters vs. 172 post-2010 -- thin
enough that it could be disproportionately driving the outlet-lean result
if pre-2010 outlets happened to have unusual lean/attribution combinations.
This re-estimates the bivariate outlet-lean model on post-2010 articles
only.


In [23]:
art_post = articles.merge(covered[['disaster_row', 'year']], on='disaster_row', how='left')
art_post = art_post[art_post['year'] >= 2010].dropna(subset=['adfontes_bias'])
print(f"Post-2010 articles with lean data: {len(art_post)} (of {articles.dropna(subset=['adfontes_bias']).shape[0]} total with lean data)")

m_post = smf.ols("attribution_score ~ adfontes_bias", data=art_post).fit(
    cov_type='cluster', cov_kwds={'groups': art_post['disaster_row']})
print(m_post.summary().tables[1])


Post-2010 articles with lean data: 779 (of 889 total with lean data)
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept         0.0665      0.037      1.811      0.070      -0.005       0.139
adfontes_bias    -0.0149      0.005     -3.250      0.001      -0.024      -0.006


### K.7 Placebo: Outlet Lean vs. Article Word Count

If outlet political lean is capturing climate-framing specifically (rather
than some generic outlet-level reporting-style difference, e.g.\ shorter
wire-style pieces vs.\ longer analytical ones), it should **not**
systematically predict something unrelated to framing, like article word
count. This is the standard placebo move in media-bias research: a clean
null here is reassuring; a significant relationship is a flag that part of
the lean effect could reflect generic style rather than climate-specific
framing.


In [24]:
word_count = []
for r in range(2, ws.max_row + 1):
    if ws.cell(row=r, column=15).value != 'Relevant':
        continue
    word_count.append(ws.cell(row=r, column=26).value)
articles['word_count'] = word_count

sub_wc = articles.dropna(subset=['adfontes_bias', 'word_count'])
print(f"N = {len(sub_wc)}")
m_placebo = smf.ols("word_count ~ adfontes_bias", data=sub_wc).fit(
    cov_type='cluster', cov_kwds={'groups': sub_wc['disaster_row']})
print(m_placebo.summary())
print("\n*** NOT a clean null: adfontes_bias is marginally significant (p < 0.10) here ***" \
      if m_placebo.pvalues['adfontes_bias'] < 0.10 else "*** Clean null, as expected for a good placebo ***")


N = 889
                            OLS Regression Results                            
Dep. Variable:             word_count   R-squared:                       0.004
Model:                            OLS   Adj. R-squared:                  0.002
Method:                 Least Squares   F-statistic:                     3.169
Date:                Mon, 22 Jun 2026   Prob (F-statistic):             0.0766
Time:                        03:26:37   Log-Likelihood:                -7218.3
No. Observations:                 889   AIC:                         1.444e+04
Df Residuals:                     887   BIC:                         1.445e+04
Df Model:                           1                                         
Covariance Type:              cluster                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept       892.1032     58.803   

### K.7b Placebo: Outlet Lean vs. Disaster Severity (Selection Into Coverage)

A second placebo, complementary to K.7: rather than testing reporting
*style* (word count), this tests *selection* -- does a more right- or
left-leaning outlet's coverage skew toward more or less severe disasters
(by NOAA-reported cost and death toll)? Cost and deaths are disaster-level
facts, not framing choices, so outlet lean should have no relationship
with them if outlets aren't simply selecting into covering systematically
different disasters by severity.


In [25]:
art_sev = articles.merge(covered[['disaster_row', 'log_cost', 'log_deaths']],
                          on='disaster_row', how='left')
sub_sev = art_sev.dropna(subset=['adfontes_bias', 'log_cost', 'log_deaths'])
print(f"N = {len(sub_sev)}")

m_placebo_cost = smf.ols("log_cost ~ adfontes_bias", data=sub_sev).fit(
    cov_type='cluster', cov_kwds={'groups': sub_sev['disaster_row']})
print("-- log(Cost) ~ outlet lean, clustered SE --")
print(m_placebo_cost.summary().tables[1])

m_placebo_deaths = smf.ols("log_deaths ~ adfontes_bias", data=sub_sev).fit(
    cov_type='cluster', cov_kwds={'groups': sub_sev['disaster_row']})
print("\n-- log(Deaths+1) ~ outlet lean, clustered SE --")
print(m_placebo_deaths.summary().tables[1])

clean_null = m_placebo_cost.pvalues['adfontes_bias'] > 0.10 and m_placebo_deaths.pvalues['adfontes_bias'] > 0.10
print("\n*** Clean null on both -- outlet lean does not predict which disasters (by severity) get covered ***"
      if clean_null else "*** Not a clean null -- see p-values above ***")


N = 889
-- log(Cost) ~ outlet lean, clustered SE --
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept         8.6251      0.202     42.799      0.000       8.230       9.020
adfontes_bias    -0.0119      0.029     -0.406      0.685      -0.069       0.046

-- log(Deaths+1) ~ outlet lean, clustered SE --


                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept         2.5313      0.240     10.562      0.000       2.062       3.001
adfontes_bias    -0.0341      0.033     -1.036      0.300      -0.099       0.030

*** Clean null on both -- outlet lean does not predict which disasters (by severity) get covered ***


### K.9 National-Only vs. Local-Only Subsamples (Effect Heterogeneity)

A different question from Section E/I (does national status itself predict
the score): are the two headline effects -- the year trend and the
outlet-lean effect -- homogeneous across national and local coverage, or
is one of them concentrated in just one segment? This re-estimates both
headline regressions separately on the national-only and local-only
article subsamples.


In [26]:
art_split = articles.merge(covered[['disaster_row', 'year_c']], on='disaster_row', how='left')

print("-- Year trend, National-only vs. Local-only --")
for label, sub in [('National', art_split[art_split.national == 1]),
                    ('Local', art_split[art_split.national == 0])]:
    m = smf.ols("attribution_score ~ year_c", data=sub).fit(
        cov_type='cluster', cov_kwds={'groups': sub['disaster_row']})
    print(f"{label}: N={len(sub)}, clusters={sub['disaster_row'].nunique()}, "
          f"b={m.params['year_c']:.4f}, p={m.pvalues['year_c']:.4f}")

print("\n-- Outlet-lean effect, National-only vs. Local-only --")
sub_lean_split = art_split.dropna(subset=['adfontes_bias'])
nat_lean_results = {}
for label, s in [('National', sub_lean_split[sub_lean_split.national == 1]),
                  ('Local', sub_lean_split[sub_lean_split.national == 0])]:
    m = smf.ols("attribution_score ~ adfontes_bias", data=s).fit(
        cov_type='cluster', cov_kwds={'groups': s['disaster_row']})
    nat_lean_results[label] = m
    print(f"{label}: N={len(s)}, clusters={s['disaster_row'].nunique()}, "
          f"b={m.params['adfontes_bias']:.4f}, p={m.pvalues['adfontes_bias']:.4f}")

m_year_nat = smf.ols("attribution_score ~ year_c", data=art_split[art_split.national == 1]).fit(
    cov_type='cluster', cov_kwds={'groups': art_split.loc[art_split.national == 1, 'disaster_row']})
m_year_loc = smf.ols("attribution_score ~ year_c", data=art_split[art_split.national == 0]).fit(
    cov_type='cluster', cov_kwds={'groups': art_split.loc[art_split.national == 0, 'disaster_row']})
m_lean_nat = nat_lean_results['National']
m_lean_loc = nat_lean_results['Local']

print("\nFull model output, National year trend:")
print(m_year_nat.summary())
print("\nFull model output, Local year trend:")
print(m_year_loc.summary())
print("\nFull model output, National outlet lean:")
print(m_lean_nat.summary())
print("\nFull model output, Local outlet lean:")
print(m_lean_loc.summary())

print("\n*** Year trend holds in BOTH segments. Outlet-lean effect is concentrated in "
      "National coverage (p=%.3f) and is NOT significant among Local-only articles (p=%.3f) -- "
      "this qualifies the lean finding: it generalizes across measurement choices (K.1-K.4) and "
      "survives disaster fixed effects, but does not appear to hold for local outlets specifically, "
      "either because the effect is genuinely concentrated in national/wire coverage or because "
      "the Local subsample (N=%d, vs. %d National) is underpowered to detect a smaller effect there. ***"
      % (m_lean_nat.pvalues['adfontes_bias'], m_lean_loc.pvalues['adfontes_bias'],
         len(sub_lean_split[sub_lean_split.national == 0]), len(sub_lean_split[sub_lean_split.national == 1])))


-- Year trend, National-only vs. Local-only --


National: N=703, clusters=186, b=0.0174, p=0.0000


Local: N=549, clusters=134, b=0.0107, p=0.0070

-- Outlet-lean effect, National-only vs. Local-only --
National: N=687, clusters=183, b=-0.0308, p=0.0007
Local: N=202, clusters=74, b=-0.0025, p=0.6892

Full model output, National year trend:
                            OLS Regression Results                            
Dep. Variable:      attribution_score   R-squared:                       0.068
Model:                            OLS   Adj. R-squared:                  0.067
Method:                 Least Squares   F-statistic:                     43.44
Date:                Mon, 22 Jun 2026   Prob (F-statistic):           4.42e-10
Time:                        03:26:37   Log-Likelihood:                -352.97
No. Observations:                 703   AIC:                             709.9
Df Residuals:                     701   BIC:                             719.1
Df Model:                           1                                         
Covariance Type:              cluster          

                            OLS Regression Results                            
Dep. Variable:      attribution_score   R-squared:                       0.021
Model:                            OLS   Adj. R-squared:                  0.019
Method:                 Least Squares   F-statistic:                     7.272
Date:                Mon, 22 Jun 2026   Prob (F-statistic):            0.00791
Time:                        03:26:37   Log-Likelihood:                -175.86
No. Observations:                 549   AIC:                             355.7
Df Residuals:                     547   BIC:                             364.3
Df Model:                           1                                         
Covariance Type:              cluster                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.1220      0.060     -2.030      0.0

### K.10 Ordered Logit/Probit (Functional-Form Check)

Every headline result so far treats $a_{i,d} \in \{-1, 0, 0.5, 1\}$ as
cardinal (equally-spaced) via OLS. That's a real assumption -- the gap
between "no claim" (C, $0$) and "hedged mention" (B, $0.5$) need not be the
same size as the gap between B and "explicit attribution" (A, $1$). This
check drops the cardinal-spacing assumption entirely and re-estimates both
headline relationships as an **ordered** outcome, via both ordered logit
and ordered probit (`statsmodels.miscmodels.ordinal_model.OrderedModel`).

The ordinal scale collapses C and E to the same level (both score $0$ and
sit at the same point in the underlying ordering: more skeptical than B,
less skeptical than... no, more anthropogenic than D, less than B): $D < \{C,E\} < B < A$,
coded $0,1,2,3$.

Caveat: `OrderedModel`'s built-in fit does not support cluster-robust
standard errors the way the OLS specifications above do, so the article-
level outlet-lean version here uses ordinary (non-clustered) MLE standard
errors. Treat the $p$-values as an approximate check on sign and rough
significance, not a like-for-like replacement for the clustered-SE
results elsewhere in this notebook.


In [27]:
from statsmodels.miscmodels.ordinal_model import OrderedModel

art_ord = articles.merge(covered[['disaster_row', 'year_c']], on='disaster_row', how='left')
ordinal_map = {'D': 0, 'C': 1, 'E': 1, 'B': 2, 'A': 3}
art_ord['ordinal_level'] = art_ord['category'].map(ordinal_map)
art_ord['ordinal_level_cat'] = pd.Categorical(art_ord['ordinal_level'], categories=[0, 1, 2, 3], ordered=True)
print(art_ord['ordinal_level'].value_counts().sort_index())

print("\n-- Ordered logit: ordinal attribution level ~ year --")
mod_logit_year = OrderedModel(art_ord['ordinal_level_cat'], art_ord[['year_c']], distr='logit')
res_logit_year = mod_logit_year.fit(method='bfgs', disp=False)
print(f"year_c: coef={res_logit_year.params['year_c']:.4f}, p={res_logit_year.pvalues['year_c']:.4g}")

print("\n-- Ordered probit: ordinal attribution level ~ year --")
mod_probit_year = OrderedModel(art_ord['ordinal_level_cat'], art_ord[['year_c']], distr='probit')
res_probit_year = mod_probit_year.fit(method='bfgs', disp=False)
print(f"year_c: coef={res_probit_year.params['year_c']:.4f}, p={res_probit_year.pvalues['year_c']:.4g}")

sub_ord = art_ord.dropna(subset=['adfontes_bias']).copy()
sub_ord['ordinal_level_cat'] = pd.Categorical(sub_ord['ordinal_level'], categories=[0, 1, 2, 3], ordered=True)

print("\n-- Ordered logit: ordinal attribution level ~ outlet lean --")
mod_logit_lean = OrderedModel(sub_ord['ordinal_level_cat'], sub_ord[['adfontes_bias']], distr='logit')
res_logit_lean = mod_logit_lean.fit(method='bfgs', disp=False)
print(f"adfontes_bias: coef={res_logit_lean.params['adfontes_bias']:.4f}, p={res_logit_lean.pvalues['adfontes_bias']:.4g}")

print("\n-- Ordered probit: ordinal attribution level ~ outlet lean --")
mod_probit_lean = OrderedModel(sub_ord['ordinal_level_cat'], sub_ord[['adfontes_bias']], distr='probit')
res_probit_lean = mod_probit_lean.fit(method='bfgs', disp=False)
print(f"adfontes_bias: coef={res_probit_lean.params['adfontes_bias']:.4f}, p={res_probit_lean.pvalues['adfontes_bias']:.4g}")

print("\nFull ordered logit (year) summary:")
print(res_logit_year.summary())
print("\nFull ordered logit (lean) summary:")
print(res_logit_lean.summary())

print("\n*** Both relationships hold under ordered logit AND ordered probit, with the same sign "
      "as the OLS specifications -- the headline results are not an artifact of treating the "
      "{-1, 0, 0.5, 1} scale as cardinal. ***")


ordinal_level
0     53
1    947
2    145
3    107
Name: count, dtype: int64

-- Ordered logit: ordinal attribution level ~ year --


year_c: coef=0.1044, p=3.438e-15

-- Ordered probit: ordinal attribution level ~ year --


year_c: coef=0.0546, p=1.36e-15

-- Ordered logit: ordinal attribution level ~ outlet lean --


adfontes_bias: coef=-0.0850, p=0.004107

-- Ordered probit: ordinal attribution level ~ outlet lean --


adfontes_bias: coef=-0.0434, p=0.006438

Full ordered logit (year) summary:
                             OrderedModel Results                             
Dep. Variable:      ordinal_level_cat   Log-Likelihood:                -973.94
Model:                   OrderedModel   AIC:                             1956.
Method:            Maximum Likelihood   BIC:                             1976.
Date:                Mon, 22 Jun 2026                                         
Time:                        03:26:37                                         
No. Observations:                1252                                         
Df Residuals:                    1248                                         
Df Model:                           1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
year_c         0.1044      0.013      7.874      0.000 

### K.11 Year x Outlet-Lean Interaction

Has the outlet-lean gap in attribution framing widened or narrowed over
the sample period? This adds an `adfontes_bias * year_c` interaction to
the article-level model, both in isolation and alongside the full set of
controls (national status, disaster type).


In [28]:
TYPE_FORM = "C(disaster_type, Treatment(reference='Severe Storm'))"

art_int = articles.merge(covered[['disaster_row', 'year_c', 'disaster_type']],
                          on='disaster_row', how='left')
sub_int = art_int.dropna(subset=['adfontes_bias'])

m_int = smf.ols("attribution_score ~ adfontes_bias * year_c", data=sub_int).fit(
    cov_type='cluster', cov_kwds={'groups': sub_int['disaster_row']})
print("-- Simple interaction model --")
print(m_int.summary())

m_int_full = smf.ols(
    "attribution_score ~ national + adfontes_bias * year_c + "
    f"{TYPE_FORM}", data=sub_int).fit(
    cov_type='cluster', cov_kwds={'groups': sub_int['disaster_row']})
print("\n-- Full model + interaction --")
print(m_int_full.summary())

b0 = m_int.params['adfontes_bias']
b1 = m_int.params['adfontes_bias:year_c']
y0, y1 = sub_int['year_c'].min(), sub_int['year_c'].max()
print(f"\nInteraction term: b={b1:.5f}, p={m_int.pvalues['adfontes_bias:year_c']:.4f} (simple model)")
print(f"Full-model interaction term: b={m_int_full.params['adfontes_bias:year_c']:.5f}, "
      f"p={m_int_full.pvalues['adfontes_bias:year_c']:.4f}")
print(f"Implied lean effect in {int(y0)+covered['year'].min()}: {b0 + b1*y0:.4f}")
print(f"Implied lean effect in {int(y1)+covered['year'].min()}: {b0 + b1*y1:.4f}")
print("\n*** Interaction is not statistically significant in either specification -- no detectable "
      "widening or narrowing of the outlet-lean gap over time. Point estimate suggests a slightly "
      "more negative (more pronounced) lean effect in later years, but this is not distinguishable "
      "from no change given the standard error. ***")


-- Simple interaction model --
                            OLS Regression Results                            
Dep. Variable:      attribution_score   R-squared:                       0.059
Model:                            OLS   Adj. R-squared:                  0.056
Method:                 Least Squares   F-statistic:                     16.66
Date:                Mon, 22 Jun 2026   Prob (F-statistic):           1.15e-09
Time:                        03:26:37   Log-Likelihood:                -434.05
No. Observations:                 889   AIC:                             876.1
Df Residuals:                     885   BIC:                             895.3
Df Model:                           3                                         
Covariance Type:              cluster                                         
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
I

### K.8 Robustness Summary

Both headline results -- the year trend and the outlet-lean effect --
survive every alternative specification and sample restriction tried here,
including the most demanding one (disaster fixed effects, K.1). The one
result that is **not** a clean pass is the placebo (K.7): outlet lean has
a marginally significant ($p \approx 0.07$-$0.10$) negative relationship
with article word count, meaning more right-leaning outlets in this
sample also tend to write somewhat shorter pieces. This doesn't overturn
the lean-attribution finding -- it survives disaster fixed effects, which
is a much harder bar -- but it means part of that relationship plausibly
reflects a generic length/depth difference across outlets rather than
climate-framing decisions specifically, and that nuance should be
acknowledged rather than treated as a clean ideological-framing result.


In [29]:
robustness_rows = [
    ("Year trend -- baseline", "b=0.0129/yr", m_time.pvalues['year_c']),
    ("Year trend -- binary DV (K.2)", f"b={m_bin_time.params['year_c']:.4f}", m_bin_time.pvalues['year_c']),
    ("Year trend -- D-subtype recoded (K.3)", f"b={m_alt_time.params['year_c']:.4f}", m_alt_time.pvalues['year_c']),
    ("Year trend -- E excluded (K.4)", f"b={m_noE_time.params['year_c']:.4f}", m_noE_time.pvalues['year_c']),
    ("Year trend -- N_d>=3 only (K.5)", f"b={m_n3.params['year_c']:.4f}", m_n3.pvalues['year_c']),
    ("Outlet lean -- baseline (clustered)", "b=-0.0132", m_lean_af.pvalues['adfontes_bias']),
    ("Outlet lean -- disaster FE (K.1)", f"b={m_fe.params['adfontes_bias_dm']:.4f}", m_fe.pvalues['adfontes_bias_dm']),
    ("Outlet lean -- binary DV (K.2)", f"b={m_bin_lean.params['adfontes_bias']:.4f}", m_bin_lean.pvalues['adfontes_bias']),
    ("Outlet lean -- D-subtype recoded (K.3)", f"b={m_alt_lean.params['adfontes_bias']:.4f}", m_alt_lean.pvalues['adfontes_bias']),
    ("Outlet lean -- E excluded (K.4)", f"b={m_noE_lean.params['adfontes_bias']:.4f}", m_noE_lean.pvalues['adfontes_bias']),
    ("Outlet lean -- post-2010 only (K.6)", f"b={m_post.params['adfontes_bias']:.4f}", m_post.pvalues['adfontes_bias']),
    ("PLACEBO: word count ~ lean (K.7)", f"b={m_placebo.params['adfontes_bias']:.4f}", m_placebo.pvalues['adfontes_bias']),
    ("PLACEBO: log(cost) ~ lean (K.7b)", f"b={m_placebo_cost.params['adfontes_bias']:.4f}", m_placebo_cost.pvalues['adfontes_bias']),
    ("PLACEBO: log(deaths+1) ~ lean (K.7b)", f"b={m_placebo_deaths.params['adfontes_bias']:.4f}", m_placebo_deaths.pvalues['adfontes_bias']),
    ("Year trend -- National-only (K.9)", f"b={m_year_nat.params['year_c']:.4f}", m_year_nat.pvalues['year_c']),
    ("Year trend -- Local-only (K.9)", f"b={m_year_loc.params['year_c']:.4f}", m_year_loc.pvalues['year_c']),
    ("Outlet lean -- National-only (K.9)", f"b={m_lean_nat.params['adfontes_bias']:.4f}", m_lean_nat.pvalues['adfontes_bias']),
    ("Outlet lean -- Local-only (K.9)", f"b={m_lean_loc.params['adfontes_bias']:.4f}", m_lean_loc.pvalues['adfontes_bias']),
    ("Year trend -- ordered logit (K.10)", f"b={res_logit_year.params['year_c']:.4f}", res_logit_year.pvalues['year_c']),
    ("Year trend -- ordered probit (K.10)", f"b={res_probit_year.params['year_c']:.4f}", res_probit_year.pvalues['year_c']),
    ("Outlet lean -- ordered logit (K.10)", f"b={res_logit_lean.params['adfontes_bias']:.4f}", res_logit_lean.pvalues['adfontes_bias']),
    ("Outlet lean -- ordered probit (K.10)", f"b={res_probit_lean.params['adfontes_bias']:.4f}", res_probit_lean.pvalues['adfontes_bias']),
    ("Outlet lean x Year interaction (K.11)", f"b={m_int.params['adfontes_bias:year_c']:.5f}", m_int.pvalues['adfontes_bias:year_c']),
]
robustness_df = pd.DataFrame(robustness_rows, columns=['Check', 'Estimate', 'p-value'])
robustness_df['Sig.'] = robustness_df['p-value'].apply(
    lambda p: '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else '')
print(robustness_df.to_string(index=False))


                                 Check    Estimate      p-value Sig.
                Year trend -- baseline b=0.0129/yr 5.390067e-06  ***
         Year trend -- binary DV (K.2)    b=0.0100 1.038079e-07  ***
 Year trend -- D-subtype recoded (K.3)    b=0.0102 3.584335e-05  ***
        Year trend -- E excluded (K.4)    b=0.0130 4.433738e-06  ***
       Year trend -- N_d>=3 only (K.5)    b=0.0112 1.594884e-08  ***
   Outlet lean -- baseline (clustered)   b=-0.0132 1.550120e-03  ***
      Outlet lean -- disaster FE (K.1)   b=-0.0115 4.638530e-02   **
        Outlet lean -- binary DV (K.2)   b=-0.0177 1.522601e-04  ***
Outlet lean -- D-subtype recoded (K.3)   b=-0.0135 7.008444e-04  ***
       Outlet lean -- E excluded (K.4)   b=-0.0135 1.396104e-03  ***
   Outlet lean -- post-2010 only (K.6)   b=-0.0149 1.154975e-03  ***
      PLACEBO: word count ~ lean (K.7)  b=-18.6898 7.504775e-02    *
      PLACEBO: log(cost) ~ lean (K.7b)   b=-0.0119 6.848533e-01     
  PLACEBO: log(deaths+1) ~ lean (K